# C11-neural-training — Practice p18 — Solution


**Type:** scenario analysis · **Difficulty:** core · **Concepts:** batch-normalization, dropout


BatchNorm weight and bias are optimizer parameters; running mean and variance
are buffers. A dropout mask is ephemeral randomness, while SGD/Adam momentum
lives in optimizer state. `no_grad()` suppresses graph recording but does not
change module mode: because the model remains in train mode, BatchNorm consumes
validation batch statistics and updates its buffers, and dropout draws a new
mask for each call. That explains both symptoms.

The lifecycle is `model.train()` for training, then `model.eval()` followed by
`with torch.no_grad():` for validation, then `model.train()` before the next
training epoch. Executable postconditions are: repeated eval logits are close
at `atol=1e-9, rtol=1e-7`; running mean and variance are unchanged; dropout is
the identity in eval mode; and `model.training` is true after restoration.
An optimizer may own BatchNorm weight and bias, but never its running buffers.


In [ ]:
import torch
import torch.nn as nn
torch.manual_seed(18); torch.set_default_dtype(torch.float64)
bn_p18=nn.BatchNorm1d(4); drop_p18=nn.Dropout(.3); X_p18=torch.randn(8,4)
bn_p18.eval(); drop_p18.eval(); before_p18=(bn_p18.running_mean.clone(),bn_p18.running_var.clone())
with torch.no_grad():
    out1_p18=drop_p18(bn_p18(X_p18)); out2_p18=drop_p18(bn_p18(X_p18)); drop_identity_p18=drop_p18(X_p18)
after_p18=(bn_p18.running_mean.clone(),bn_p18.running_var.clone()); bn_p18.train(); drop_p18.train()


### Answer check


In [ ]:
assert torch.allclose(out1_p18,out2_p18,atol=1e-9,rtol=1e-7)
assert all(torch.equal(a,b) for a,b in zip(before_p18,after_p18))
drop_p18.eval(); assert torch.allclose(drop_p18(X_p18),X_p18,atol=1e-9,rtol=1e-7)
bn_p18.train(); drop_p18.train(); assert bn_p18.training and drop_p18.training
assert {name for name,_ in bn_p18.named_parameters()}=={"weight","bias"}
assert {name for name,_ in bn_p18.named_buffers()}=={"running_mean","running_var","num_batches_tracked"}
